# 说明
本代码原本需要 Azure AI Foundry 和 Azure AI Agent Service，在Azure云服务上配置，注册账号需要绑定VIS卡，有兴趣的可以自己试试，说面在下个单元格中。由于windows+jupyter有兼容性问题，用需要用py方式执行代码，11-mcp.py是把大模型调用换成Azure提供免费的LLM调用服务后的演示代码

### 核心演示内容
1. **MCP协议集成**
    - 通过MCPStdioPlugin连接到真实的OpenBnB MCP服务器
    - 使用标准输入输出(stdio)与外部服务通信
    - 自动发现MCP服务器提供的可用功能
2. **真实数据访问**
    - 搜索真实的Airbnb房源信息（非模拟数据）
    - 获取包括价格、评分、评论等详细信息
    - 将结果以HTML表格形式美观展示
3. **生产级代理构建**
    - 使用Azure OpenAI作为LLM后端
    - 实现错误处理和环境验证
    - 创建专业化的Agent指令和响应格式

### 为什么这个功能重要
MCP（模型上下文协议）是AI代理领域的重要技术，它解决了以下关键问题：
- **标准化工具集成**：提供统一方式让AI代理调用外部服务
- **降低开发门槛**：无需为每个服务编写特定集成代码
- **增强实用性**：让AI代理能访问真实世界数据，而不仅是模拟数据
- **促进生态系统**：不同服务可以遵循同一协议，便于互操作

# 使用 Semantic Kernel 集成 OpenBnB MCP 服务器

本笔记展示了如何使用 Semantic Kernel 与实际的 OpenBnB MCP 服务器集成，通过 MCPStdioPlugin 搜索真实的 Airbnb住宿。对于 LLM 访问，它使用了 Azure AI Foundry。要设置您的环境变量，可以参考 [设置课程](https://github.com/microsoft/ai-agents-for-beginners/blob/main/translations/zh/00-course-setup/README.md)。


## 导入所需的包


In [1]:
# Import cell - Updated imports
# 导入必要的库 - 就像做饭前准备好所有食材

# 在文件开头添加（在其他导入之前）
import nest_asyncio
nest_asyncio.apply()

import json  # 用于处理JSON数据格式
import os    # 用于操作系统相关功能，如读取环境变量
import asyncio  # 用于异步编程，让程序能同时做多件事
import subprocess  # 用于运行外部命令（如npx）
import sys  # 提供对Python解释器的访问

# 从dotenv导入load_dotenv，用于加载.env文件中的环境变量
from dotenv import load_dotenv
# 用于在Jupyter Notebook中显示HTML内容
from IPython.display import display, HTML
# 用于给函数参数添加描述信息
from typing import Annotated


from openai import AsyncOpenAI

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion  # OpenAI连接器
# 从semantic_kernel导入关键组件
from semantic_kernel.agents import (  # Agent相关组件
    ChatCompletionAgent,     # 基于聊天完成的Agent
    ChatHistoryAgentThread   # 用于维护对话历史的线程
)
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion  # Azure OpenAI连接器
from semantic_kernel.connectors.mcp import MCPStdioPlugin  # MCP协议插件
from semantic_kernel.contents import (  # 不同类型的内容对象
    FunctionCallContent,     # 表示函数调用的内容
    FunctionResultContent,   # 表示函数调用结果的内容
    StreamingTextContent     # 表示流式文本响应的内容
)

## 创建 MCP 插件连接

我们将使用 MCPStdioPlugin 连接到 [OpenBnB MCP 服务器](https://github.com/openbnb-org/mcp-server-airbnb)。该服务器通过 @openbnb/mcp-server-airbnb 包提供 Airbnb 搜索功能。


## 创建客户端

在本示例中，我们将使用 Azure AI Foundry 来访问 LLM。请确保您的环境变量已正确设置。


## 环境配置

配置 Azure OpenAI 设置。确保已设置以下环境变量：
- `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`


In [ ]:
# Creating the Client cell - Updated for Azure
load_dotenv()

# Azure OpenAI configuration
# Ensure these environment variables are set:
# - AZURE_OPENAI_CHAT_DEPLOYMENT_NAME
# - AZURE_OPENAI_ENDPOINT
# - AZURE_OPENAI_API_KEY (optional if using DefaultAzureCredential)

# 初始化一个基于 Azure (微软云) OpenAI 的聊天补全服务对象
chat_completion_service = AzureChatCompletion(
    
    # 部署名称 (Deployment Name)：
    # 在 Azure OpenAI 中，你需要先“部署”一个模型（比如 GPT-4）。这个名字就是你在控制台设定的部署名。
    # 这里通过 os.getenv 从环境变量中读取，避免把敏感信息硬编码在代码里。
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    
    # 终结点 (Endpoint)：
    # 这是 Azure 为你分配的专属 API 请求地址，通常长得像：https://<你的资源名>.openai.azure.com/
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    
    # API 密钥 (API Key)：
    # 用于身份验证的秘钥。
    # 注释提示：如果你在 Azure 中配置了基于身份（DefaultAzureCredential）的无密钥认证，这个参数是可选的。
    # 否则，它也会从系统环境变量里读取你的密钥。
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

## 了解 OpenBnB MCP 集成

此笔记本连接到**真实的 OpenBnB MCP 服务器**，提供实际的 Airbnb 搜索功能。

### 工作原理：

1. **MCPStdioPlugin**：通过标准输入/输出与 MCP 服务器进行通信  
2. **真实的 NPM 包**：通过 npx 下载并运行 `@openbnb/mcp-server-airbnb`  
3. **实时数据**：从 Airbnb 的 API 返回真实的房源数据  
4. **功能发现**：代理会自动发现 MCP 服务器提供的可用功能  

### 可用功能：

OpenBnB MCP 服务器通常提供以下功能：  
- **search_listings** - 根据位置和条件搜索 Airbnb 房源  
- **get_listing_details** - 获取特定房源的详细信息  
- **check_availability** - 检查特定日期的房源可用性  
- **get_reviews** - 获取房源的评论  
- **get_host_info** - 获取房源主人的信息  

### 前置条件：

- 系统已安装 **Node.js**  
- **互联网连接**，用于下载 MCP 服务器包  
- **NPX** 可用（随 Node.js 一起安装）  

### 测试连接：

您可以通过运行以下命令手动测试 MCP 服务器：  
```bash
npx -y @openbnb/mcp-server-airbnb
```  

此命令将下载并启动 OpenBnB MCP 服务器，随后 Semantic Kernel 会连接到该服务器以获取真实的 Airbnb 数据。  


## 运行与 OpenBnB MCP 服务器连接的代理

现在我们将运行与 OpenBnB MCP 服务器连接的 AI 代理，用于搜索斯德哥尔摩适合2名成人和1名儿童的真实Airbnb住宿。您可以随意修改 `user_inputs` 列表来更改搜索条件。


In [13]:
user_inputs = [
    "Find Airbnb in Stockholm for 2 adults 1 kid",
]

# 主函数：运行MCP启用的代理，使用Azure OpenAI与OpenBnB服务器通信
async def main():
    """Main function to run the MCP-enabled agent with real OpenBnB server using Azure OpenAI"""

    try:
        print("🚀 Starting with Azure OpenAI...")
        
        # Verify environment variables
        print("🔍 Checking Azure environment variables...")
        # 必需的环境变量列表
        required_vars = [
            "AZURE_OPENAI_CHAT_DEPLOYMENT_NAME",  # 部署名称
            "AZURE_OPENAI_ENDPOINT",             # Azure服务端点
            "AZURE_OPENAI_API_KEY"               # API密钥
        ]
        
        # 检查每个环境变量是否存在
        for var in required_vars:
            if os.getenv(var):
                print(f"✅ {var} is set")
            else:
                print(f"❌ {var} is NOT set")
        
        # 开始创建MCP插件连接
        print("\n🔧 Creating MCP Plugin...")
        
        # Create MCP plugin connection to real OpenBnB server
        # Based on the GitHub repo, the server doesn't need special env vars
        # 使用async with确保资源正确清理
        # MCPStdioPlugin通过标准输入输出与MCP服务器通信
        async with MCPStdioPlugin(
            name="AirbnbSearch",  # 插件名称
            description="Search for Airbnb accommodations using OpenBnB MCP server",  # 插件描述
            command="npx",  # 要运行的命令
            args=["-y", "@openbnb/mcp-server-airbnb"],  # 命令参数：下载并运行Airbnb MCP服务器
        ) as airbnb_plugin:  # 将插件命名为airbnb_plugin

            # MCP插件已创建并连接
            print("✅ MCP Plugin created and connected")
            
            # Wait a moment for the server to fully initialize
            # 等待服务器初始化（给服务器一点时间启动）
            await asyncio.sleep(2)
            
            # Try to list available tools
            # 尝试列出可用工具（查看MCP服务器提供哪些功能）
            try:
                tools = await airbnb_plugin.get_tools()
                print(f"🔧 Available tools: {[tool.name for tool in tools]}")
            except Exception as e:
                print(f"⚠️ Could not list tools: {str(e)}")

            # 创建Azure OpenAI服务连接
            # Create the Azure OpenAI service with proper configuration
            print("\n🤖 Creating Azure OpenAI service...")
            # service = AzureChatCompletion(
            #     deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
            #     endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            #     api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            # )

            # 使用AsyncOpenAI客户端创建服务实例
            model_name = "gpt-4o-mini"
            client = AsyncOpenAI(
                api_key=os.environ["GITHUB_TOKEN"],
                base_url="https://models.inference.ai.azure.com/"
            )
            service = OpenAIChatCompletion(
                # 指定要使用的模型ID（这里是通义千问的qwen-max）
                ai_model_id=model_name,
                # 传入之前创建的AsyncOpenAI客户端
                async_client=client,
            )

            
            # Create agent with the service instance
            # 创建AI代理
            # 指令翻译如下：
            # 你是一个Airbnb搜索助手。使用可用功能搜索房源。
            # 将结果格式化为清晰的HTML表格，包含房源名称、价格、评分和链接。
            agent = ChatCompletionAgent(
                service=service,  # 使用上面创建的Azure服务
                name="AirbnbAgent",  # 代理名称
                # 代理指令 - 告诉AI如何行为
                instructions="""You are an Airbnb search assistant. Use the available functions to search for properties. 
                Format results in a clear HTML table with columns for property name, price, rating, and link.""",
                plugins=[airbnb_plugin],
            )

            print("✅ Agent created with Azure OpenAI")

            # Process each user input
            # 创建线程来保存对话历史
            thread: ChatHistoryAgentThread | None = None

            # 依次处理每个用户输入
            for user_input in user_inputs:
                print(f"\n🔍 User: {user_input}")
                
                try:
                    # Use the simpler get_response method
                    # 让代理处理用户输入
                    response = await agent.get_response(messages=user_input, thread=thread)
                    thread = response.thread  # 更新对话线程
                    
                    # 获取响应文本
                    response_text = str(response)
                    
                    # 移除可能的Markdown代码块标记
                    response_text = response_text.replace('```html', '').replace('```', '')
                    
                    # 打印响应摘要
                    print(f"🤖 {response.name}: {response_text[:200]}..." if len(response_text) > 200 else response_text)
                    
                    # If response contains HTML table, display it properly
                    # 如果响应包含HTML表格，添加CSS样式并显示
                    if '<table' in response_text.lower():
                        # Add CSS styling for better table rendering
                        table_css = """
                        <style>
                            .airbnb-results table {
                                border-collapse: collapse;
                                width: 100%;
                                margin: 10px 0;
                            }
                            .airbnb-results th, .airbnb-results td {
                                border: 1px solid #ddd;
                                padding: 8px;
                                text-align: left;
                            }
                            .airbnb-results th {
                                background-color: #f2f2f2;
                                font-weight: bold;
                            }
                            .airbnb-results tr:nth-child(even) {
                                background-color: #f9f9f9;
                            }
                            .airbnb-results a {
                                color: #1976d2;
                                text-decoration: none;
                            }
                            .airbnb-results a:hover {
                                text-decoration: underline;
                            }
                        </style>
                        """
                        html_output = f'{table_css}<div class="airbnb-results">{response_text}</div>'
                        display(HTML(html_output))
                    else:
                        # Display as regular text if no table
                        display(HTML(f'<div class="airbnb-results">{response_text}</div>'))
                        
                except Exception as e:
                    print(f"❌ Error processing user input: {str(e)}")
                    import traceback
                    traceback.print_exc()
                
            # Cleanup
            # 清理对话线程
            if thread:
                await thread.delete()
                print("🧹 Thread cleaned up")
                    
    except Exception as e:
        print(f"❌ Main error: {str(e)}")
        import traceback
        traceback.print_exc()

# Run the main function
print("🚀 Starting MCP Agent...")
await main()
print("✅ Done!")

🚀 Starting MCP Agent...
🚀 Starting with Azure OpenAI...
🔍 Checking Azure environment variables...
✅ AZURE_OPENAI_CHAT_DEPLOYMENT_NAME is set
✅ AZURE_OPENAI_ENDPOINT is set
✅ AZURE_OPENAI_API_KEY is set

🔧 Creating MCP Plugin...
✅ MCP Plugin created and connected
⚠️ Could not list tools: 'MCPStdioPlugin' object has no attribute 'get_tools'

🤖 Creating Azure OpenAI service...
✅ Agent created with Azure OpenAI

🔍 User: Find Airbnb in Stockholm for 2 adults 1 kid
Please provide me with the specific dates you plan to stay in Stockholm, and I'll help you find suitable Airbnb properties!


🧹 Thread cleaned up
❌ Main error: Failed to connect to the MCP server. Please check your configuration.
✅ Done!


Traceback (most recent call last):
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\site-packages\mcp\os\win32\utilities.py", line 169, in create_windows_process
    process = await anyio.open_process(
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\site-packages\anyio\_core\_subprocesses.py", line 190, in open_process
    return await get_async_backend().open_process(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\site-packages\anyio\_backends\_asyncio.py", line 2588, in open_process
    process = await asyncio.create_subprocess_exec(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\asyncio\subprocess.py", line 224, in create_subprocess_exec
    transport, protocol = await loop.subprocess_exec(
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\asyncio\bas

# 概要
恭喜你！你已经成功构建了一个能够通过模型上下文协议（MCP）与现实世界住宿搜索集成的 AI 代理：

## 使用的技术：
- Semantic Kernel - 用于使用 Azure OpenAI 构建智能代理
- Azure AI Foundry - 提供大语言模型功能和聊天补全
- MCP（模型上下文协议） - 用于标准化工具集成
- OpenBnB MCP Server - 提供真实的 Airbnb 搜索功能
- Node.js/NPX - 用于运行外部 MCP 服务器

## 你学到了什么：
- MCP 集成：将 Semantic Kernel 代理连接到外部 MCP 服务器
- 实时数据访问：通过实时 API 搜索实际的 Airbnb 房源
- 协议通信：使用标准输入输出（stdio）在代理和 MCP 服务器之间通信
- 功能发现：自动发现 MCP 服务器提供的可用功能
- 流式响应：实时捕获并记录函数调用
- HTML 渲染：以样式化表格和交互式显示格式化代理响应

## 下一步：
- 集成更多 MCP 服务器（如天气、航班、餐厅）
- 构建结合 MCP 和 A2A 协议的多代理系统
- 为你的数据源创建自定义 MCP 服务器
- 实现跨会话的持久对话记忆
- 将代理部署到 Azure Functions，并实现 MCP 服务器编排
- 添加用户认证和预订功能



---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。应以原始语言的文档作为权威来源。对于重要信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
